# Session 11: Backpropagation by hand

**KLAi | Mathematics for AI and Machine Learning**

In this lab you run the forward pass, the backward pass and the update on the tiny network from the slides (2 inputs, 2 hidden units with a sigmoid, 1 output, squared-error loss). Then you prove your nine gradients with finite differences, and you train the network for 50 steps. The same numbers are in the spreadsheet lab. The network and its numbers are illustrative (made-up numbers to show the arithmetic).

Each step has the same rhythm: **PREDICT**, run, **READ**, then **CHANGE one thing**. A cell that says `None  # your code here` is yours to fill. The check cells say OK or NOT YET.

In [ ]:
# Automatic check helper. Run this cell once. You do not need to read it.
_results = {}   # label -> True or False; running a check again replaces its old result
def check(label, value, expected, tol=0.01):
    try:
        import numpy as _np
        ok = bool(_np.all(_np.abs(_np.asarray(value, dtype=float) - _np.asarray(expected, dtype=float)) <= tol))
    except Exception:
        ok = False
    _results[label] = ok
    print(("OK       " if ok else "NOT YET  ") + label)
def show_checks(total=None):
    # total = how many checks this notebook has; a skipped or crashed cell still counts as NOT YET
    k = sum(_results.values()); n = max(total or 0, len(_results))
    print("All %d checks pass" % n if k == n and n else "%d of %d checks pass" % (k, n))

---
## Step 1: The network and the forward pass

The inputs are x = (1, 2) and the target is y = 1. W1 is 2 × 2 (one row per hidden unit), W2 is 1 × 2. The forward pass is: z(1) = W(1) x + b(1), then a(1) = sigmoid(z(1)), then y-hat = W(2) a(1) + b(2), then L = 0.5 (y-hat - y)^2. The layer is written in brackets. In the code the same objects are called z1, a1, W1, b1, W2 and b2, and the deltas d1 and d2.

**PREDICT first.** Is the prediction y-hat going to be bigger or smaller than the target 1? Write your guess before you run.

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x = np.array([1.0, 2.0])               # the two inputs
y = 1.0                                # the target
W1 = np.array([[0.2, -0.3], [0.4, 0.1]])   # weights into the hidden units, one row per hidden unit
b1 = np.array([0.1, -0.2])             # biases of the hidden units
W2 = np.array([[0.5, -0.4]])           # weights into the output unit (1 × 2)
b2 = np.array([0.2])                   # bias of the output unit
eta = 0.5                              # the learning rate

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

In [ ]:
# Hint: use the forward pass formulas in the step text; @ is matrix multiplication; the output unit gives a one-entry array, so take its entry to get a number
z1 = None    # your code here: the two weighted sums of the hidden units, plus their biases
a1 = None    # your code here: the sigmoid of z1
yhat = None  # your code here: the prediction, one number
L = None     # your code here: half the squared error

In [ ]:
print("z1 =", z1)
print("a1 =", a1)
print("y-hat =", yhat, " loss =", L)
check("z1 (both hidden units)", z1, [-0.3000, 0.4000], 0.001)
check("a1 (both hidden units)", a1, [0.4256, 0.5987], 0.001)
check("prediction y-hat", yhat, 0.1733, 0.001)
check("loss L", L, 0.3417, 0.001)

**READ the output.** The prediction is far below the target. Which way must most of the parameters move to raise it?

> My answer: 

---
## Step 2: The backward pass: deltas and the nine gradients

delta(2) = y-hat - y. Then delta(1) = (W(2) transposed times delta(2)) times a(1) times (1 - a(1)), entry by entry. The gradient of a weight matrix is the delta times the transposed values that enter it. The gradient of a bias is its delta.

**PREDICT first.** Will delta(2) be positive or negative? Will the gradients of W(2) have the same sign as delta(2)?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
# Hint: d2 has shape (1,) and d1 has shape (2,). Each weight gradient is a matrix with the same shape as its weights; np.outer builds a matrix from two vectors.
d2 = None   # your code here: the prediction minus the target, as an array with one entry
d1 = None   # your code here: send d2 back through W2 transposed, then multiply entry by entry by the sigmoid slope of a1; shape (2,)
gW2 = None  # your code here: delta(2) times the transposed hidden outputs; shape (1, 2)
gb2 = None  # your code here: a bias gradient equals its delta
gW1 = None  # your code here: delta(1) times the transposed inputs; shape (2, 2)
gb1 = None  # your code here: a bias gradient equals its delta

In [ ]:
print("delta2 =", d2, " delta1 =", d1)
print("gradient of W2:", gW2)
print("gradient of W1:", gW1)
check("delta2", d2, -0.8267, 0.001)
check("delta1 (two values)", d1, [-0.1010, 0.0794], 0.001)
check("gradient of W2", gW2, [[-0.3518, -0.4949]], 0.001)
check("gradient of W1", gW1, [[-0.1010, -0.2021], [0.0794, 0.1589]], 0.001)

**READ the output.** Print the shape of each gradient with .shape. Is each gradient the same shape as its parameter?

> My answer: 

---
## Step 3: Prove the gradients with finite differences

To trust the nine gradients, nudge each parameter by +h and -h (h = 0.00001), compute the loss both times, and divide the difference by 2h. The helper below packs the nine parameters into one list so a loop can nudge them one at a time. Fill in the answer cell of Step 2 first, because this step uses its gradients.

**PREDICT first.** How close do you expect the two answers to be: within 0.1, within 0.001, or within 0.0000001?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
def loss_of(p):
    """The loss for a list of the nine parameters: W1 (4 numbers), b1 (2), W2 (2), b2 (1)."""
    W1_, b1_, W2_, b2_ = p[0:4].reshape(2, 2), p[4:6], p[6:8].reshape(1, 2), p[8:9]
    a_ = sigmoid(W1_ @ x + b1_)
    return 0.5 * ((W2_ @ a_ + b2_)[0] - y) ** 2

p0 = np.concatenate([W1.ravel(), b1, W2.ravel(), b2])             # the nine parameters
h = 0.00001
numeric = np.zeros(9)
for i in range(9):
    up, down = p0.copy(), p0.copy()
    up[i] += h
    down[i] -= h
    numeric[i] = (loss_of(up) - loss_of(down)) / (2 * h)
print("finite differences:", np.round(numeric, 6))
if any(g is None for g in (gW1, gb1, gW2, gb2)):
    backprop = None
    print("Fill in the answer cell of step 2 first: its nine gradients are compared here.")
else:
    backprop = np.concatenate([gW1.ravel(), gb1, gW2.ravel(), gb2])   # the nine gradients from step 2
    print("backpropagation:   ", np.round(backprop, 6))

In [ ]:
# Hint: the largest absolute difference between the two lists of nine numbers (np.abs, then np.max)
max_diff = None  # your code here

In [ ]:
print("largest difference:", max_diff)
check("finite differences agree with backpropagation (difference below 1e-7)", max_diff, 0.0, 1e-7)

**CHANGE one thing.** Break one formula on purpose: in step 2 delete the factor a1 * (1 - a1) from d1, re-run steps 2 and 3, and see which entries of the check disagree. Then put the factor back and re-run.

> What I saw: 

---
## Step 4: The update

Every parameter moves by minus eta times its gradient. Then we run the forward pass again with the new parameters.

**PREDICT first.** Will the loss after one update be about the same, a little smaller, or much smaller than 0.3417?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
# Hint: apply the update rule to each of the four groups of parameters
W1n = None  # your code here
b1n = None   # your code here
W2n = None   # your code here
b2n = None   # your code here

In [ ]:
if any(v is None for v in (W1n, b1n, W2n, b2n)):
    W1n_row1, loss1 = None, None
    print("Fill in the four updates above first.")
else:
    yhat1 = (W2n @ sigmoid(W1n @ x + b1n) + b2n)[0]
    loss1 = 0.5 * (yhat1 - y) ** 2
    W1n_row1 = W1n[0]
check("new W1 row 1", W1n_row1, [0.2505, -0.1990], 0.001)
check("new b2", b2n, 0.6133, 0.001)
check("loss after one update", loss1, 0.0085, 0.0005)
if loss1 is not None:
    print("new W1 row 1:", W1n_row1, " new loss:", loss1)

**CHANGE one thing.** Change eta to 0.1, re-run step 4 and read the new loss. Is a smaller learning rate better or worse here? Put eta back to 0.5 afterwards.

> What I saw: 

---
## Step 5: Repeat: 50 training steps

The function below is the whole training step (forward, backward, update) in a loop. Read it line by line before you run it: it is the code from the slides.

In [ ]:
def train(eta, steps):
    W1_, b1_, W2_, b2_ = W1.copy(), b1.copy(), W2.copy(), b2.copy()
    losses = []
    for step in range(steps + 1):
        a_ = sigmoid(W1_ @ x + b1_)                 # forward pass
        yh_ = W2_ @ a_ + b2_
        losses.append(0.5 * (yh_[0] - y) ** 2)
        d2_ = yh_ - y                               # backward pass
        d1_ = (W2_.T @ d2_) * a_ * (1 - a_)
        W2_, b2_ = W2_ - eta * np.outer(d2_, a_), b2_ - eta * d2_     # update
        W1_, b1_ = W1_ - eta * np.outer(d1_, x), b1_ - eta * d1_
    return losses

losses = train(0.5, 50)
print("loss at steps 0, 1, 2, 3:", [round(float(l), 6) for l in losses[:4]])
plt.plot(losses)
plt.yscale("log")
plt.xlabel("step")
plt.ylabel("loss (log scale)")
plt.title("Training the tiny network, eta = 0.5")
plt.show()

**PREDICT first.** On the log scale the loss falls by a factor that changes over the first few steps and then settles. Near which factor does it settle: 2, 10 or 60?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
# Hint: losses holds the loss at step 0, 1, 2 and so on; pick the entries you need
loss2 = None    # your code here: loss after two updates
loss50 = None   # your code here: loss after 50 updates

In [ ]:
print(loss2, loss50)
check("loss after two updates", loss2, 0.000157, 0.00002)
check("loss after 50 updates is almost zero (below 1e-6)", loss50, 0.0, 1e-6)

**READ the output.** The prediction for these inputs is now very close to 1. Does this mean the network is good in general? (It has seen one example only.)

> My answer: 

---
## Step 6: A learning rate that is too large

Run the same training with eta = 5. You know from Session 10 what to expect. The check records the loss after just one update.

**PREDICT first.** With eta = 5, will the loss after one update be smaller or larger than at the start (0.3417)?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
# Hint: train returns a list of losses; pick the entry after one update
loss_big = None  # your code here: the loss after one update with eta = 5

In [ ]:
print("loss at the start 0.3417, loss after one update with eta = 5:", loss_big)
check("loss after one update with eta = 5", loss_big, 16.2753, 0.01)

**CHANGE one thing.** Call train(5.0, 5) and print the losses (train takes its own eta, so nothing needs undoing). How does the loss behave from step to step? Then call train(0.05, 50) and compare its last loss with the eta = 0.5 run.

> What I saw: 

---
## Finish

Run the cell below. It should say **All 15 checks pass**. Then restart the kernel, run all cells from the top, and save the notebook.

In [ ]:
show_checks(15)